<a href="https://colab.research.google.com/github/nieblasIX/IX-Pachuca-RESISTENCIATUZA/blob/main/ResistenciaTuzaPachuca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cartografía de lo Invisible: Resistencia de la Tuza en Pachuca**

Cambiar la mirada de la tuza
un animal históricamente desplazado y visto como plaga para convertirlo en un indicador de justicia espacial y de memoria histórica en Pachuca es un ejercicio de Filosofía de la Liberación aplicada a la ecología. Pachuca es una ciudad construida sobre túneles mineros de extracción humana; las tuzas construyen túneles para la regeneración del suelo. Es una perfecta ironía poética y espacial.

**Planteamiento Ecológico y de Liberación: El Ingeniero del Suelo**

Ecológicamente, los geómidos como las tuzas(Cratogeomys spp.) no son plagas, son ingenieros del ecosistema. Remueven el suelo, permiten la aireación, aumentan la infiltración de agua y entierran materia orgánica. Su exterminio en Pachuca no es un accidente biológico, sino el resultado de un modelo urbanístico que valora el suelo sólo como superficie de tránsito o construcción, ignorando el volumen biológico del subsuelo.

**El Conflicto:** La tuza requiere suelos profundos, aireados y con cobertura vegetal (raíces/bulbos). La urbanización impone sellado asfáltico y compactación, lo que equivale a la "desaparición" de su mundo tridimensional.

**La Perspectiva de la Liberación:** Siguiendo a Dussel, situamos a la tuza como el "Sujeto Oprimido" del ecosistema urbano. Su "plaga" es, en realidad, un acto de presencia frente al despojo de su hábitat histórico por la lógica del capital minero y ahora inmobiliario.

 La tuza representa a la Alteridad oprimida por la totalidad del sistema urbano-industrial. Mapear su hábitat no es solo un ejercicio técnico, es un acto de justicia interespecie: visibilizar al que el sistema ha decidido ignorar y destruir.

**Variables Satelitales Clave**

Para un análisis de series de tiempo (pensemos en una escala de 20 a 30 años con Landsat 5, 7, 8 y 9, complementado con Sentinel-2), las variables deben medir tres cosas: *comida (vegetación), casa (suelo) y amenaza (urbanización).*
1. **Dinámica de la Vegetación (Alimento y Cobertura)**
NDVI / EVI: Tendencias anuales y estacionales. Las tuzas prefieren pastizales y zonas de matorral con raíces palatables. Un análisis armónico de NDVI en GEE te dirá si la fenología del paisaje ha cambiado.
2. **Estado del Suelo
(El Medio Físico)**: BSI (Bare Soil Index) Para identificar zonas de suelo desnudo o degradado por la expansión minera y urbana; asi como un Índice de Humedad de Diferencia Normalizada (NDMI). Las tuzas evitan suelos saturados de agua (se ahogan) pero necesitan suelo que no sea roca pura para excavar.
3. **La Huella Antrópica(La Opresión)** con
LST (Land Surface Temperature): Las islas de calor urbanas degradan el microclima del suelo. También el NDBI (Índice de Edificación) / Luces Nocturnas (VIIRS/DMSP)que servirán para mapear el crecimiento de la mancha urbana que fragmenta el territorio año con año.
4. **Topografía y termodinámica (El Escenario)**: Usando el MDE (modelo digital de elevación SRTM o NASADEM) para encontrar pendientes y orientación. Las tuzas prefieren pendientes suaves donde el drenaje del suelo sea eficiente. Asímismo, las tuzas son sensibles al estrés térmico. El efecto "Isla de Calor" en Pachuca limita sus corredores que se observaría con el cálculo de LST (Land Surface Temperature).

**Análisis Temporal: La Crónica del Despojo**

Proponemos un análisis de tres nodos temporales (e.g., 1990, 2000, 2012-2026):

**Detección de Cambio de Cobertura**(LULC): Cuantificar cuántas hectáreas de suelo apto han sido "selladas".

**Análisis de Fragmentación:** Utilizar métricas de ecología del paisaje (a través de Python/Geemap) para calcular el aislamiento de las poblaciones de tuzas en "islas" urbanas (baldíos, camellones, campos de fútbol).

**Construcción del Modelo de Hábitat de la Tuza**

Para modelar el hábitat sin caer en determinismos absurdos, podemos construir un Índice de Idoneidad de Hábitat (HSI) o correr un modelo de clasificación automática (Random Forest) basado en presencia-pseudoausencia.
El Índice de Idoneidad de Hábitat (HSI) será nuestra herramienta de denuncia. No es solo un mapa de probabilidad, es un mapa de derecho al territorio.



In [2]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

# Autenticación (Te pedirá iniciar sesión con tu cuenta de Google)
ee.Authenticate()
ee.Initialize(project='resistenciatuza') # Sustituye si tienes un ID de proyecto, o borra el parámetro project

# 2. DEFINICIÓN DEL TERRITORIO Y TOPOGRAFÍA
pachuca = ee.FeatureCollection("FAO/GAUL/2015/level2") \
            .filter(ee.Filter.eq('ADM2_NAME', 'Pachuca de Soto'))

# Topografía (SRTM 30m)
srtm = ee.Image('USGS/SRTMGL1_003')
elevacion = srtm.select('elevation').clip(pachuca)
pendiente = ee.Terrain.slope(elevacion)

# 3. FUNCIÓN PARA LST Y ALBEDO (Landsat 8 - Ejemplo para 2026)
# El cálculo de LST requiere conversiones de radiancia y emisividad.
# Usaremos un producto de Landsat que ya trae la temperatura superficial (Collection 2 Level 2).
l8_2026 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(pachuca) \
    .filterDate('2025-01-01', '2026-03-01') \
    .filter(ee.Filter.lt('CLOUD_COVER', 10)) \
    .median().clip(pachuca)

# La banda ST_B10 es la temperatura en Kelvin. Convertimos a Celsius.
lst_celsius = l8_2026.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST')

# Albedo aproximado (Fórmula de Liang para Landsat)
albedo = l8_2026.expression(
    '(0.356 * B2) + (0.130 * B4) + (0.373 * B5) + (0.085 * B6) + (0.072 * B7) - 0.0018', {
        'B2': l8_2026.select('SR_B2').multiply(0.0000275).add(-0.2), # Blue
        'B4': l8_2026.select('SR_B4').multiply(0.0000275).add(-0.2), # Red
        'B5': l8_2026.select('SR_B5').multiply(0.0000275).add(-0.2), # NIR
        'B6': l8_2026.select('SR_B6').multiply(0.0000275).add(-0.2), # SWIR1
        'B7': l8_2026.select('SR_B7').multiply(0.0000275).add(-0.2)  # SWIR2
}).rename('Albedo')

# 4. TASA DE CRECIMIENTO URBANO (Análisis Temporal 1990-2026)
# Para cuantificar en qué años hubo más despojo, extraemos el promedio anual del NDBI.
def get_annual_ndbi(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)

    # Filter for Landsat 5
    col_l5 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2") \
        .filterBounds(pachuca) \
        .filterDate(start, end) \
        .filter(ee.Filter.lt('CLOUD_COVER', 10))

    # Filter for Landsat 8
    col_l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterBounds(pachuca) \
        .filterDate(start, end) \
        .filter(ee.Filter.lt('CLOUD_COVER', 10))

    # Select the appropriate collection based on the year
    current_col = ee.Algorithms.If(
        ee.Number(year).gte(2013),
        col_l8,
        col_l5
    )
    current_col = ee.ImageCollection(current_col) # Ensure it's an ImageCollection type

    # Check if the collection is empty
    is_empty = current_col.size().eq(0)

    # Calculate NDBI image if collection is not empty
    ndbi_image = ee.Algorithms.If(
        is_empty,
        ee.Image(0), # Return a dummy image (with all zeros) if empty
        ee.Algorithms.If(
            ee.Number(year).gte(2013),
            current_col.median().normalizedDifference(['SR_B6', 'SR_B5']),
            current_col.median().normalizedDifference(['SR_B5', 'SR_B4'])
        )
    )

    # Ensure ndbi_image is an ee.Image and rename the band
    ndbi_image = ee.Image(ndbi_image).rename('NDBI')

    # Reduce region to get the mean NDBI
    mean_ndbi_value = ndbi_image \
        .reduceRegion(reducer=ee.Reducer.mean(), geometry=pachuca.geometry(), scale=30, maxPixels=1e9) \
        .get('NDBI')

    return ee.Feature(None, {'year': year, 'mean_NDBI': mean_ndbi_value})

# Mapear la función sobre los años
years = ee.List.sequence(1990, 2026)
annual_ndbi_fc = ee.FeatureCollection(years.map(get_annual_ndbi))

# Extraer a un DataFrame de Python para graficar la "Tasa de Despojo"
ndbi_data = geemap.ee_to_df(annual_ndbi_fc)

# 5. EL PLAN B: EXPORTACIÓN PARA QGIS (Para evitar saturar la memoria)
# Si quieres mandar el raster de Temperatura de 2026 directo a tu Drive para abrirlo en QGIS:
task = ee.batch.Export.image.toDrive(
    image=lst_celsius,
    description='LST_Pachuca_2026',
    folder='GEE_Tuza',
    scale=30,
    region=pachuca.geometry(),
    maxPixels=1e10
)
task.start()
print("Exportación de Temperatura a Google Drive iniciada. Revisa tu Drive en unos minutos.")

Exportación de Temperatura a Google Drive iniciada. Revisa tu Drive en unos minutos.


In [1]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo para que parezca reporte académico/profesional
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))

# Graficar la tendencia del NDBI
sns.lineplot(data=ndbi_data, x='year', y='mean_NDBI', marker='o', color='#e63946', linewidth=2.5)

# Añadir títulos con la narrativa del proyecto
plt.title('Crónica del Despojo: Crecimiento de la Mancha Urbana en Pachuca (1990-2026)', fontsize=16, pad=20)
plt.xlabel('Año', fontsize=12)
plt.ylabel('Índice de Edificación Promedio (NDBI)', fontsize=12)

# Resaltar hitos (Aquí puedes ajustar los años según lo que veas en la gráfica)
plt.axvspan(2010, 2015, color='gray', alpha=0.2, label='Expansión Zona Plateada (aprox)')
plt.axvline(x=2024, color='black', linestyle='--', label='Inicio Obras Tren/Megaproyectos')

plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'ndbi_data' is not defined

<Figure size 1200x600 with 0 Axes>

In [ ]:
display(ndbi_data.head())

NameError: name 'ndbi_data' is not defined